# 01 Data Integrity (E-01 to E-04)

**Data used:** the full raw file, target-free (FR-009).  
**Questions:** Q1 shape and types, Q2 allowed values, Q3 key integrity, Q4 parsing contract (DOC-02 §4.1).

All computation lives in `house_price.eda` (FR-010): this notebook only runs it and shows the saved outputs (`reports/eda/`, `reports/figures/eda/`).

In [1]:
from IPython.display import Image, Markdown, display

from house_price.eda import EDAContext
from house_price.eda import integrity

ctx = EDAContext.load()  # validated raw file + persisted development set (never the held-out rows)

In [2]:
outputs = integrity.run(ctx)
sorted(outputs.tables) + sorted(outputs.figures)

['E-01_shape_schema',
 'E-02_allowed_values_audit',
 'E-02_value_counts',
 'E-03_key_integrity',
 'E-04_parsing_comparison']

## E-01 Shape and schema: every column with its declared contract and loaded dtype.

In [3]:
outputs.tables["E-01_shape_schema"]

,column,source_name,role,schema_dtype,loaded_dtype,dtype_matches,nullable,n_missing
0,Id,Order,identifier,int,int64,True,False,0
1,PID,PID,identifier,string,str,True,False,0
2,MSSubClass,MS SubClass,model_input,int,int64,True,False,0
3,MSZoning,MS Zoning,model_input,category,str,True,False,0
4,LotFrontage,Lot Frontage,model_input,float,float64,True,True,490
...,...,...,...,...,...,...,...,...
77,MoSold,Mo Sold,model_input,int,int64,True,False,0
78,YrSold,Yr Sold,model_input,int,int64,True,False,0
79,SaleType,Sale Type,excluded,category,str,True,False,0
80,SaleCondition,Sale Condition,excluded,category,str,True,False,0


## E-02 Allowed-values audit: observed values versus allowed codes; `not_allowed` must be empty.

In [4]:
outputs.tables["E-02_allowed_values_audit"]

,column,n_allowed,n_observed,not_allowed,allowed_unobserved,note
0,MSSubClass,16,16,,,"Dwelling-class code, stored zero-padded in the..."
1,MSZoning,11,7,,A|C|I|RP,Includes the file's spellings that differ from...
2,Street,2,2,,,
3,Alley,2,2,,,
4,LotShape,4,4,,,
5,LandContour,4,4,,,
6,Utilities,4,3,,ELO,
7,LotConfig,5,5,,,
8,LandSlope,3,3,,,
9,Neighborhood,29,28,,Names,Includes the file's spellings that differ from...


In [5]:
outputs.tables["E-02_value_counts"]

,column,value,count,allowed
0,Alley,<missing>,2732,True
1,Alley,Grvl,120,True
2,Alley,Pave,78,True
3,BldgType,1Fam,2425,True
4,BldgType,TwnhsE,233,True
...,...,...,...,...
295,Street,Pave,2918,True
296,Street,Grvl,12,True
297,Utilities,AllPub,2927,True
298,Utilities,NoSewr,2,True


## E-03 Key integrity: identifiers, `SalePrice > 0`, raw-file hash.

In [6]:
outputs.tables["E-03_key_integrity"]

,check,result,detail
0,row_count,2930,rows in the raw file
1,column_count,82,columns in the raw file
2,Id_unique,True,0 duplicate value(s)
3,PID_unique,True,0 duplicate value(s)
4,SalePrice_positive,True,0 value(s) <= 0
5,raw_sha256_confirmed,True,65a1cbb89c2b58b11674097135dd04e710df5db726bcfe...


## E-04 Parsing comparison: explicit tokens (DN-18) versus library defaults. `MasVnrType` shows the hazard.

In [7]:
outputs.tables["E-04_parsing_comparison"].query("difference != 0")

,column,explicit_missing,default_missing,difference
26,MasVnrType,23,1775,1752
